# 🤖 Algoritmos de Clustering: K-Means y DBSCAN
### Aprendizaje No Supervisado — UNAM AI Lab
---
> **Archivo:** `21_UNAM_AI_Actividad_Laboratorio_4_2_clase.ipynb`  
> **Tema:** Algoritmos de Clustering — Aprendizaje No Supervisado  
> **Contenido:** Fundamentos matemáticos, descripción formal, pseudocódigo, implementación y buenas prácticas


## 📋 Tabla de Contenidos
1. [Introducción al Aprendizaje No Supervisado](#intro)
2. [K-Means](#kmeans)
   - Fundamentos matemáticos
   - Descripción narrativa y formal
   - Pseudocódigo
   - Implementación con buenas prácticas
   - Métricas de complejidad
3. [DBSCAN](#dbscan)
   - Fundamentos matemáticos
   - Descripción narrativa y formal
   - Pseudocódigo
   - Implementación con buenas prácticas
   - Métricas de complejidad
4. [Comparación K-Means vs DBSCAN](#comparacion)

---
## 1. Introducción al Aprendizaje No Supervisado <a id='intro'></a>

En el **Aprendizaje No Supervisado** no contamos con etiquetas $y$ que guíen el entrenamiento. El reto es encontrar **estructuras ocultas** en los datos basándose únicamente en su similitud o distribución.

El **Clustering** es la tarea principal: agrupar puntos similares entre sí y distintos de otros grupos. Se utiliza en:
- Segmentación de clientes
- Detección de anomalías
- Clasificación de imágenes médicas
- Agrupación de documentos
- Detección de minas terrestres en campos de batalla


In [ ]:
# ============================================================
# IMPORTACIONES — Buena práctica: todas las importaciones al inicio
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler

# Semilla para reproducibilidad — Buena práctica
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Librerías importadas correctamente.")

---
## 2. K-Means <a id='kmeans'></a>

K-Means es un algoritmo de aprendizaje **no supervisado** que se utiliza cuando se tienen datos sin etiquetar. **El objetivo es encontrar grupos en los datos; los puntos se agrupan según la similitud de características.**

### 2.1 Fundamentos Matemáticos

K-Means busca minimizar la **Inercia** (WCSS — Within-Cluster Sum of Squares):

$$J = \sum_{j=1}^{K} \sum_{x_i \in C_j} \| x_i - \mu_j \|^2$$

donde:
- $K$ = número de clusters
- $C_j$ = conjunto de puntos en el cluster $j$
- $\mu_j$ = centroide del cluster $j$
- $\| x_i - \mu_j \|^2$ = distancia euclidiana al cuadrado

**Distancia Euclidiana** (métrica estándar):
$$d(x, \mu) = \sqrt{\sum_{i}(x_i - \mu_i)^2}$$

**Actualización de centroides** (promedio de los puntos asignados):
$$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$

### 2.2 Descripción Narrativa

Imagina una nube de puntos en un plano que quieres dividir en $K$ grupos:

- **Centroides:** Cada grupo tiene un "representante" llamado centroide, que es el centro geométrico de sus puntos.
- **Asignación:** Cada punto decide unirse al grupo cuyo centroide esté más cerca (por distancia euclidiana).
- **Actualización:** Una vez que todos los puntos eligieron bando, los centroides se mueven al promedio real de sus nuevos miembros.
- **Iteración:** Se repite hasta que los centroides dejen de moverse. Es como un baile donde los líderes buscan el centro de su gente y la gente sigue a su líder más cercano.

### 2.3 Descripción Formal del Algoritmo

**Pasos:**
1. Agrupa los datos en $K$ grupos, donde $K$ está predefinido.
2. Selecciona $K$ puntos al azar como centros de grupo.
3. Asigna cada objeto a su centro de clúster más cercano según la distancia.
4. Calcula el centroide (media) de todos los objetos en cada grupo.
5. Repite los pasos 2–4 hasta asignar los mismos puntos a cada grupo en rondas consecutivas.

### 2.4 Métricas de Complejidad

| Métrica | Complejidad |
|---------|------------|
| **Tiempo (entrenamiento)** | $O(I \cdot K \cdot n \cdot d)$ |
| **Espacio** | $O((n + K) \cdot d)$ |

donde $I$ = iteraciones hasta convergencia, $K$ = clusters, $n$ = muestras, $d$ = dimensiones.

### 2.5 Propiedades clave
1. **Convergencia:** Garantiza converger a un mínimo, aunque puede ser un mínimo **local**. Por eso es común correrlo varias veces con diferentes inicios aleatorios.
2. **Sensibilidad:** Es muy sensible a los valores atípicos (outliers) y a la escala de los datos (¡hay que normalizar!).


### 2.6 Pseudocódigo

```
Algoritmo: K-means Clustering
Entrada:  Dataset X, número de clusters K
Salida:   K centroides y etiquetas de pertenencia de cada punto

1. Inicializar K centroides aleatoriamente (pueden ser puntos del mismo dataset)
2. REPETIR hasta que no haya cambios (o se llegue a un máximo de iteraciones):
   a. PASO DE ASIGNACIÓN:
      Para cada punto x_i:
          Encontrar el centroide μ_j más cercano
          Asignar x_i al clúster C_j
   b. PASO DE ACTUALIZACIÓN:
      Para cada clúster C_j:
          Calcular el nuevo centroide:
          μ_j = (1 / |C_j|) * Σ x_i  para x_i ∈ C_j
3. Retornar centroides y etiquetas
```

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: K-Means desde cero
# Buenas prácticas aplicadas:
#   - Docstrings en cada método
#   - Nombres descriptivos de variables
#   - Constantes en MAYÚSCULAS
#   - Manejo de casos borde (cluster vacío)
#   - Separación de responsabilidades (un método = una tarea)
# ============================================================

class KMeans:
    """Implementación de K-Means Clustering desde cero.
    
    Parámetros
    ----------
    K : int
        Número de clusters deseados.
    max_iters : int
        Número máximo de iteraciones antes de detener el algoritmo.
    """

    def __init__(self, K: int = 3, max_iters: int = 100):
        self.K = K
        self.max_iters = max_iters
        self.centroids = None
        self.clusters = [[] for _ in range(self.K)]

    def fit(self, X: np.ndarray) -> np.ndarray:
        """Entrena el modelo K-Means sobre el dataset X.
        
        Parámetros
        ----------
        X : np.ndarray de forma (n_samples, n_features)
        
        Retorna
        -------
        labels : np.ndarray con la etiqueta de cluster de cada punto.
        """
        self.X = X
        self.n_samples, self.n_features = X.shape

        # Paso 1: Inicializar centroides aleatoriamente (tomando puntos del dataset)
        random_sample_idxs = np.random.choice(self.n_samples, self.K, replace=False)
        self.centroids = self.X[random_sample_idxs]

        for _ in range(self.max_iters):
            # Paso 2a: Asignación — crear clusters basados en el centroide más cercano
            self.clusters = self._create_clusters(self.centroids)

            # Guardar centroides viejos para checar convergencia
            centroids_old = self.centroids.copy()

            # Paso 2b: Actualización — calcular nuevos centroides (promedio de los puntos)
            self.centroids = self._get_centroids(self.clusters)

            # Si los centroides no cambian, terminamos (convergencia)
            if self._is_converged(centroids_old, self.centroids):
                break

        return self._get_cluster_labels(self.clusters)

    def _create_clusters(self, centroids: np.ndarray) -> list:
        """Asigna cada punto al centroide más cercano."""
        clusters = [[] for _ in range(self.K)]
        for idx, sample in enumerate(self.X):
            # Calculamos distancia euclidiana a todos los centroides
            distances = [np.sqrt(np.sum((sample - c) ** 2)) for c in centroids]
            # Nos quedamos con el índice del más cercano
            closest_idx = np.argmin(distances)
            clusters[closest_idx].append(idx)
        return clusters

    def _get_centroids(self, clusters: list) -> np.ndarray:
        """Calcula el nuevo centroide como el promedio de los puntos en cada cluster."""
        centroids = np.zeros((self.K, self.n_features))
        for cluster_idx, cluster in enumerate(clusters):
            if not cluster:  # Evitar división por cero si un cluster queda vacío
                centroids[cluster_idx] = self.X[np.random.choice(self.n_samples)]
            else:
                centroids[cluster_idx] = np.mean(self.X[cluster], axis=0)
        return centroids

    def _is_converged(self, old_centroids: np.ndarray, new_centroids: np.ndarray) -> bool:
        """Verifica si los centroides dejaron de moverse (convergencia)."""
        distances = [np.sqrt(np.sum((old - new) ** 2))
                     for old, new in zip(old_centroids, new_centroids)]
        return sum(distances) == 0

    def _get_cluster_labels(self, clusters: list) -> np.ndarray:
        """Convierte la lista de clusters a un arreglo de etiquetas por punto."""
        labels = np.empty(self.n_samples)
        for cluster_idx, cluster in enumerate(clusters):
            for sample_idx in cluster:
                labels[sample_idx] = cluster_idx
        return labels


print("✅ Clase KMeans definida correctamente.")

In [ ]:
# ============================================================
# APLICACIÓN Y VISUALIZACIÓN DE K-MEANS
# ============================================================

# Generar datos sintéticos con 3 clusters bien separados
N_CLUSTERS = 3
N_SAMPLES = 300

X_blobs, y_true = make_blobs(
    n_samples=N_SAMPLES,
    centers=N_CLUSTERS,
    cluster_std=0.8,
    random_state=RANDOM_SEED
)

# Normalizar — Buena práctica indispensable
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_blobs)

# Entrenar K-Means
km = KMeans(K=N_CLUSTERS, max_iters=100)
labels = km.fit(X_scaled)

# ---- Visualización ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Datos originales sin etiquetar
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c='gray', s=30, alpha=0.6)
axes[0].set_title('Datos originales (sin etiquetar)', fontsize=13)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Resultado del clustering
colors = ['#E74C3C', '#2ECC71', '#3498DB']
for k in range(N_CLUSTERS):
    mask = labels == k
    axes[1].scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                    c=colors[k], s=30, alpha=0.7, label=f'Cluster {k+1}')

# Dibujar centroides
axes[1].scatter(km.centroids[:, 0], km.centroids[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroides')
axes[1].set_title('Resultado K-Means (K=3)', fontsize=13)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.suptitle('K-Means Clustering', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Centroides finales:\n{km.centroids}")

In [ ]:
# ============================================================
# MÉTODO DEL CODO — Elegir el K óptimo
# ============================================================

def calcular_inercia(X: np.ndarray, labels: np.ndarray, centroids: np.ndarray) -> float:
    """Calcula la inercia (WCSS) del clustering resultante."""
    inercia = 0.0
    for k, centroid in enumerate(centroids):
        puntos_cluster = X[labels == k]
        inercia += np.sum((puntos_cluster - centroid) ** 2)
    return inercia


inercias = []
rango_k = range(1, 9)

for k in rango_k:
    modelo = KMeans(K=k, max_iters=100)
    etiquetas = modelo.fit(X_scaled)
    inercia = calcular_inercia(X_scaled, etiquetas, modelo.centroids)
    inercias.append(inercia)

plt.figure(figsize=(8, 4))
plt.plot(rango_k, inercias, marker='o', color='#3498DB', linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', label='K óptimo = 3')
plt.title('Método del Codo — Selección de K', fontsize=13)
plt.xlabel('Número de clusters K')
plt.ylabel('Inercia (WCSS)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. DBSCAN <a id='dbscan'></a>
### *Density-Based Spatial Clustering of Applications with Noise*

A diferencia de K-Means, **DBSCAN no necesita que le digamos cuántos grupos hay** y, lo más importante, **no asume que los grupos son círculos**. Puede encontrar formas de "luna", "anillos" y, sobre todo, detectar **ruido** (outliers), algo que K-Means simplemente no puede hacer.

### 3.1 Fundamentos Matemáticos

DBSCAN ve el mundo como **densidades**. Dados un conjunto de puntos $D$ y parámetros $\varepsilon$ (epsilon) y $MinPts$:

**Vecindad $\varepsilon$** de un punto $p$:
$$N_\varepsilon(p) = \{ q \in D \mid dist(p, q) \leq \varepsilon \}$$

**Tipos de puntos:**
- **Core Point (Punto Núcleo):** $|N_\varepsilon(p)| \geq MinPts$ — tiene suficientes vecinos cercanos.
- **Border Point (Punto Frontera):** No es Core Point, pero está en la vecindad de uno.
- **Noise/Outlier (Ruido):** No es Core Point ni está cerca de uno.

**Alcanzabilidad por densidad:**  
Un punto $q$ es **directamente alcanzable** desde $p$ si $p$ es Core Point y $q \in N_\varepsilon(p)$.  
Un punto $q$ es **alcanzable por densidad** desde $p$ si existe una cadena $p_1, ..., p_n$ tal que $p_1 = p$, $p_n = q$ y cada $p_{i+1}$ es directamente alcanzable desde $p_i$.

**Un Cluster** es el conjunto **máximo** de puntos alcanzables por densidad entre sí.

### 3.2 Descripción Narrativa

Imagina que estás en una fiesta:
- **Puntos Núcleo (Core Points):** Son personas en el centro de un grupo denso (tienen al menos $MinPts$ amigos cerca).
- **Puntos Frontera (Border Points):** Tienen pocos amigos cerca, pero al menos uno de ellos es un "Punto Núcleo".
- **Ruido (Noise/Outliers):** Son personas que están solas y no tienen a ningún "Punto Núcleo" cerca.

El algoritmo expande los grupos conectando Puntos Núcleo que están a una distancia máxima $\varepsilon$ entre sí.

### 3.3 Parámetros clave
1. **$\varepsilon$ (Epsilon):** Define el radio de la vecindad. Si es muy pequeño → todo será ruido. Si es muy grande → todos los puntos se unirán en un solo grupo.
2. **$MinPts$:** Define la "masa crítica" para considerar una región como densa. Regla común: $MinPts \geq d + 1$ donde $d$ es la dimensión.
3. **No paramétrico en K:** DBSCAN descubre el número de clústeres automáticamente.

### 3.4 Métricas de Complejidad

| Métrica | Complejidad |
|---------|------------|
| **Tiempo (promedio)** | $O(n \log n)$ con KD-Trees |
| **Tiempo (peor caso)** | $O(n^2)$ con matriz de distancias completa |
| **Espacio** | $O(n)$ para almacenar etiquetas y datos |


### 3.5 Pseudocódigo

```
Algoritmo: DBSCAN
Entrada:  Dataset X, ε (epsilon), MinPts
Salida:   Etiquetas de cluster por punto (-1 = ruido)

1. Etiquetar todos los puntos como NO VISITADOS.
2. Para cada punto P en X:
   a. Si P ya fue visitado → continuar al siguiente.
   b. Marcar P como VISITADO.
   c. Buscar vecinos de P a distancia ≤ ε.
   d. Si cantidad de vecinos < MinPts:
         Marcar P como RUIDO (label = -1).
   e. Sino (P es Core Point):
         Crear un nuevo Cluster C y añadir P.
         EXPANDIR CLUSTER:
           Para cada vecino P' de P:
             Si P' no fue visitado:
               Marcar P' como VISITADO.
               Buscar vecinos de P' a distancia ≤ ε.
               Si cantidad de vecinos ≥ MinPts:
                 Añadir sus vecinos a la cola de expansión.
             Si P' no pertenece a ningún cluster:
               Añadir P' al Cluster C.
3. Retornar etiquetas.
```

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: DBSCAN desde cero
# Buenas prácticas aplicadas:
#   - Docstrings completos
#   - Constante NOISE claramente definida
#   - Separación de responsabilidades
#   - Cola de expansión eficiente
#   - Nombres descriptivos
# ============================================================

class DBSCAN:
    """Implementación de DBSCAN desde cero.

    Parámetros
    ----------
    eps : float
        Radio máximo de la vecindad (epsilon).
    min_samples : int
        Número mínimo de puntos para considerar un Core Point.
    """

    NOISE = -1  # Etiqueta para puntos clasificados como ruido

    def __init__(self, eps: float = 0.5, min_samples: int = 5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels = None

    def fit(self, X: np.ndarray) -> np.ndarray:
        """Ejecuta el algoritmo DBSCAN sobre el dataset X.

        Parámetros
        ----------
        X : np.ndarray de forma (n_samples, n_features)

        Retorna
        -------
        labels : np.ndarray con la etiqueta de cluster de cada punto.
                 -1 indica ruido (outlier).
        """
        n_samples = X.shape[0]
        # Inicializar: todos los puntos marcados como ruido (-1)
        self.labels = np.full(n_samples, self.NOISE)
        visited = np.zeros(n_samples, dtype=bool)
        cluster_id = 0

        for i in range(n_samples):
            if visited[i]:
                continue

            visited[i] = True
            neighbors = self._get_neighbors(X, i)

            if len(neighbors) < self.min_samples:
                # Punto con pocos vecinos → ruido (puede cambiar si lo alcanza un Core Point)
                self.labels[i] = self.NOISE
            else:
                # Core Point → expandir un nuevo cluster
                self._expand_cluster(X, i, neighbors, cluster_id, visited)
                cluster_id += 1

        return self.labels

    def _get_neighbors(self, X: np.ndarray, target_idx: int) -> np.ndarray:
        """Devuelve los índices de todos los puntos dentro de la vecindad ε del punto target_idx."""
        # Calculamos distancia euclidiana a todos los demás puntos
        distances = np.linalg.norm(X - X[target_idx], axis=1)
        return np.where(distances <= self.eps)[0]

    def _expand_cluster(
        self,
        X: np.ndarray,
        root_idx: int,
        neighbors: np.ndarray,
        cluster_id: int,
        visited: np.ndarray
    ) -> None:
        """Expande el cluster desde el Core Point root_idx hacia todos sus vecinos alcanzables."""
        self.labels[root_idx] = cluster_id

        # Usamos una lista dinámica para simular la cola de expansión
        queue = list(neighbors)

        while queue:
            current_p = queue.pop(0)

            if not visited[current_p]:
                visited[current_p] = True
                current_neighbors = self._get_neighbors(X, current_p)

                # Si current_p también es Core Point, expandimos desde él
                if len(current_neighbors) >= self.min_samples:
                    queue.extend(current_neighbors)

            # Asignar al cluster si aún no pertenece a uno
            if self.labels[current_p] == self.NOISE:
                self.labels[current_p] = cluster_id


print("✅ Clase DBSCAN definida correctamente.")

In [ ]:
# ============================================================
# APLICACIÓN Y VISUALIZACIÓN DE DBSCAN
# Datos con forma de lunas — K-Means fallaría aquí
# ============================================================

# Generar datos en forma de dos lunas (K-Means no puede con esto)
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=RANDOM_SEED)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

# Aplicar DBSCAN
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels_db = dbscan.fit(X_moons_scaled)

# Aplicar K-Means sobre los mismos datos (para comparar)
km_moons = KMeans(K=2, max_iters=100)
labels_km = km_moons.fit(X_moons_scaled)

# ---- Visualización ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means resultado
colors_km = ['#E74C3C', '#3498DB']
for k in range(2):
    mask = labels_km == k
    axes[0].scatter(X_moons_scaled[mask, 0], X_moons_scaled[mask, 1],
                    c=colors_km[k], s=30, alpha=0.7, label=f'Cluster {k+1}')
axes[0].set_title('K-Means (K=2) — Falla con formas no esféricas', fontsize=11)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()

# DBSCAN resultado
unique_labels = np.unique(labels_db)
palette = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']
for idx, label in enumerate(unique_labels):
    mask = labels_db == label
    color = 'black' if label == -1 else palette[idx % len(palette)]
    lbl = 'Ruido (outlier)' if label == -1 else f'Cluster {label + 1}'
    axes[1].scatter(X_moons_scaled[mask, 0], X_moons_scaled[mask, 1],
                    c=color, s=30, alpha=0.7, label=lbl)
axes[1].set_title('DBSCAN (ε=0.3, MinPts=5) — Detecta formas arbitrarias', fontsize=11)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.suptitle('Comparación: K-Means vs DBSCAN en datos tipo "luna"',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

n_clusters_encontrados = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_ruido = np.sum(labels_db == -1)
print(f"DBSCAN encontró {n_clusters_encontrados} cluster(s) y {n_ruido} punto(s) de ruido.")

---
## 4. Comparación K-Means vs DBSCAN <a id='comparacion'></a>

| Característica | K-Means | DBSCAN |
|---|---|---|
| **Requiere K** | ✅ Sí | ❌ No |
| **Forma de clusters** | Solo esférica | Cualquier forma |
| **Detecta outliers** | ❌ No | ✅ Sí |
| **Sensible a escala** | ✅ Sí | ✅ Sí |
| **Complejidad tiempo** | $O(I \cdot K \cdot n \cdot d)$ | $O(n \log n)$ promedio |
| **Complejidad espacio** | $O((n+K) \cdot d)$ | $O(n)$ |
| **Convergencia** | Garantizada (mínimo local) | No aplica |
| **Parámetros** | K | $\varepsilon$, MinPts |
| **Cuándo usarlo** | Clusters compactos y bien separados | Formas arbitrarias, con ruido |


In [ ]:
# ============================================================
# VISUALIZACIÓN FINAL — Comparación completa en 4 escenarios
# ============================================================
from sklearn.datasets import make_circles

# Datasets
X_blobs_s = StandardScaler().fit_transform(
    make_blobs(n_samples=200, centers=3, cluster_std=0.6, random_state=RANDOM_SEED)[0])
X_moons_s = StandardScaler().fit_transform(
    make_moons(n_samples=200, noise=0.07, random_state=RANDOM_SEED)[0])
X_circles_s = StandardScaler().fit_transform(
    make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=RANDOM_SEED)[0])

datasets = [
    ('Blobs (globos)', X_blobs_s, 3, 0.4, 4),
    ('Lunas', X_moons_s, 2, 0.3, 5),
    ('Círculos concéntricos', X_circles_s, 2, 0.3, 5),
]

fig, axes = plt.subplots(len(datasets), 2, figsize=(12, 12))

for row, (name, X, k, eps, minpts) in enumerate(datasets):
    # K-Means
    km_ = KMeans(K=k, max_iters=100)
    lkm = km_.fit(X)
    palette = ['#E74C3C', '#2ECC71', '#3498DB']
    for ki in range(k):
        mask = lkm == ki
        axes[row, 0].scatter(X[mask, 0], X[mask, 1],
                             c=palette[ki], s=25, alpha=0.7)
    axes[row, 0].scatter(km_.centroids[:, 0], km_.centroids[:, 1],
                         c='black', marker='X', s=150, zorder=5)
    axes[row, 0].set_title(f'K-Means — {name}', fontsize=10)
    axes[row, 0].set_xlabel('Feature 1')
    axes[row, 0].set_ylabel('Feature 2')

    # DBSCAN
    db_ = DBSCAN(eps=eps, min_samples=minpts)
    ldb = db_.fit(X)
    unique = np.unique(ldb)
    for label in unique:
        mask = ldb == label
        c = 'black' if label == -1 else palette[label % len(palette)]
        lbl = 'Ruido' if label == -1 else f'Cluster {label+1}'
        axes[row, 1].scatter(X[mask, 0], X[mask, 1],
                             c=c, s=25, alpha=0.7, label=lbl)
    axes[row, 1].set_title(f'DBSCAN — {name}', fontsize=10)
    axes[row, 1].set_xlabel('Feature 1')
    axes[row, 1].set_ylabel('Feature 2')
    axes[row, 1].legend(fontsize=7)

plt.suptitle('K-Means vs DBSCAN — Comparación en distintos tipos de datos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("\n✅ Comparación completa generada.")
print("Conclusión: DBSCAN supera a K-Means en formas no esféricas y detecta outliers.")
print("K-Means es preferible cuando los clusters son compactos, esféricos y K es conocido.")